In [4]:
import pandas as pd

df = pd.read_csv('paysim.csv')
print(df.shape)
print(df['type'].value_counts())
print(df.groupby('type')['isFraud'].sum())

(6362620, 11)
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64
type
CASH_IN        0
CASH_OUT    4116
DEBIT          0
PAYMENT        0
TRANSFER    4097
Name: isFraud, dtype: int64


In [5]:
print(df.head(5))

   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  


In [6]:
SEED = 12345  # cố định seed — tái hiện đúng tinh thần "reproducible" của bài viết

fraud_df = df[df['isFraud'] == 1]
non_fraud_df = df[df['isFraud'] == 0].sample(n=250000, random_state=SEED)
sample_df = pd.concat([fraud_df, non_fraud_df]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Tổng số dòng sau khi sample: {len(sample_df)}")
print(f"Số dòng fraud: {sample_df['isFraud'].sum()}")

Tổng số dòng sau khi sample: 258213
Số dòng fraud: 8213


In [7]:
def classify_account(account_id):
    return 'Merchant' if account_id.startswith('M') else 'Client'

all_accounts = pd.unique(sample_df[['nameOrig', 'nameDest']].values.ravel())
account_df = pd.DataFrame({'id': all_accounts})
account_df['type'] = account_df['id'].apply(classify_account)

print(account_df['type'].value_counts())
client_accounts = account_df[account_df['type'] == 'Client']['id'].tolist()
merchant_accounts = account_df[account_df['type'] == 'Merchant']['id'].tolist()

type
Client      392614
Merchant     84071
Name: count, dtype: int64


In [8]:
# Tìm các Client có liên quan đến giao dịch fraud (đóng vai trò nạn nhân hoặc mule)
fraud_related_clients = pd.unique(
    sample_df[sample_df['isFraud'] == 1][['nameOrig', 'nameDest']].values.ravel()
)
fraud_related_clients = [c for c in fraud_related_clients if c.startswith('C')]

print(f"Số Client liên quan đến fraud: {len(fraud_related_clients)}")

Số Client liên quan đến fraud: 16382


In [9]:
from faker import Faker
import random

Faker.seed(SEED)
random.seed(SEED)
fake = Faker()

def generate_identity():
    return {
        'ssn': fake.ssn(),
        'email': fake.email(),
        'phone': fake.phone_number()
    }

# Sinh một pool identity riêng biệt, đủ dùng cho phần lớn Client bình thường
n_normal_clients = len(client_accounts) - len(fraud_related_clients)
identity_pool = [generate_identity() for _ in range(n_normal_clients)]

In [10]:
account_identity_map = {}

# 7a. Client bình thường: mỗi người 1 identity riêng
normal_clients = [c for c in client_accounts if c not in fraud_related_clients]
for i, client_id in enumerate(normal_clients):
    account_identity_map[client_id] = identity_pool[i % len(identity_pool)]

# 7b. Client liên quan fraud: chia thành các "cụm" nhỏ dùng chung 1 identity
# -> mô phỏng đúng ý tưởng "1st Party Fraudster tạo nhiều Mule từ 1 bộ identity bị đánh cắp"
random.shuffle(fraud_related_clients)
cluster_size = 4  # trung bình mỗi fraudster tạo ra ~4 mule dùng chung identity

for i in range(0, len(fraud_related_clients), cluster_size):
    cluster = fraud_related_clients[i:i + cluster_size]
    shared_identity = generate_identity()  # 1 identity dùng chung cho cả cụm
    for client_id in cluster:
        account_identity_map[client_id] = shared_identity

print(f"Tổng số Client đã gán identity: {len(account_identity_map)}")

Tổng số Client đã gán identity: 392614


In [11]:
identity_records = [
    {'account_id': acc, **identity}
    for acc, identity in account_identity_map.items()
]
identity_df = pd.DataFrame(identity_records)
identity_df.to_csv('identities.csv', index=False)

sample_df.to_csv('paysim_sample.csv', index=False)

print("Đã lưu paysim_sample.csv và identities.csv")
print(identity_df.head())

Đã lưu paysim_sample.csv và identities.csv
    account_id          ssn                        email                phone
0   C664893350  427-94-0167        melissa34@example.net  (825)416-4829x82518
1   C586239708  152-27-1276       marymorris@example.com   408-860-0292x41619
2   C778364717  239-68-9765        lindsay31@example.com         482.649.6820
3   C118681401  886-02-2463  michaelpeterson@example.com   200-207-1620x53817
4  C1924466574  642-63-5338      jlivingston@example.org    (618)421-7253x322


In [12]:
# Kiểm tra 1: Không có Merchant nào bị gán identity nhầm
merchant_in_identity = identity_df[identity_df['account_id'].str.startswith('M')]
assert len(merchant_in_identity) == 0, "LỖI: có Merchant bị gán identity!"

# Kiểm tra 2: Xác nhận có tồn tại các identity dùng chung (đúng mục đích thiết kế)
shared_ssn = identity_df.groupby('ssn').size()
shared_ssn = shared_ssn[shared_ssn > 1]
print(f"Số SSN được dùng chung bởi >1 account: {len(shared_ssn)}")
print(f"Tổng số account liên quan đến SSN dùng chung: {shared_ssn.sum()}")

# Kiểm tra 3: Đối chiếu các account dùng chung SSN có thực sự đều là fraud-related không
shared_ssn_accounts = identity_df[identity_df['ssn'].isin(shared_ssn.index)]['account_id'].tolist()
overlap = set(shared_ssn_accounts) & set(fraud_related_clients)
print(f"Tỷ lệ account dùng chung SSN mà đúng là fraud-related: {len(overlap)}/{len(shared_ssn_accounts)}")

Số SSN được dùng chung bởi >1 account: 4180
Tổng số account liên quan đến SSN dùng chung: 16554
Tỷ lệ account dùng chung SSN mà đúng là fraud-related: 16382/16554


### Reduce file Indentities.csv

In [13]:
import pandas as pd

identity_df = pd.read_csv('identities.csv')

# Giữ 100% Client fraud-related (không được cắt)
fraud_related_identities = identity_df[identity_df['account_id'].isin(fraud_related_clients)]

# Chỉ lấy mẫu một phần Client bình thường — giảm mạnh số dòng
normal_identities = identity_df[~identity_df['account_id'].isin(fraud_related_clients)]
normal_sample = normal_identities.sample(n=50000, random_state=SEED)  # giảm từ ~376k xuống 50k

reduced_identity_df = pd.concat([fraud_related_identities, normal_sample])
reduced_identity_df.to_csv('identities_reduced.csv', index=False)

print(f"Số dòng sau khi giảm: {len(reduced_identity_df)}")  # ~66,382 thay vì 392,614

Số dòng sau khi giảm: 66382


In [14]:
from neo4j import GraphDatabase
import os, getpass

# Thông tin kết nối lấy từ biến môi trường, không hardcode trong notebook.
# Đặt trước khi chạy:
#   Windows (PowerShell):  $env:NEO4J_PASSWORD = "..."
#   macOS / Linux:         export NEO4J_PASSWORD="..."
# Nếu chưa đặt, notebook sẽ hỏi mật khẩu bằng getpass (không lưu vào output).
URI  = os.getenv("NEO4J_URI",  "neo4j://127.0.0.1:7687")
USER = os.getenv("NEO4J_USER", "neo4j")
PWD  = os.getenv("NEO4J_PASSWORD") or getpass.getpass("Neo4j password: ")

driver = GraphDatabase.driver(URI, auth=(USER, PWD))

# Kiểm tra kết nối trước khi làm gì khác — tránh debug mù nếu sai password/port
driver.verify_connectivity()
print("Kết nối Neo4j thành công")

Kết nối Neo4j thành công


In [15]:
def create_constraints(tx):
    constraints = [
        "CREATE CONSTRAINT client_id IF NOT EXISTS FOR (c:Client) REQUIRE c.id IS UNIQUE",
        "CREATE CONSTRAINT merchant_id IF NOT EXISTS FOR (m:Merchant) REQUIRE m.id IS UNIQUE",
        "CREATE CONSTRAINT tx_id IF NOT EXISTS FOR (t:Transaction) REQUIRE t.id IS UNIQUE",
        "CREATE CONSTRAINT ssn_unique IF NOT EXISTS FOR (s:SSN) REQUIRE s.ssn IS UNIQUE",
        "CREATE CONSTRAINT email_unique IF NOT EXISTS FOR (e:Email) REQUIRE e.email IS UNIQUE",
        "CREATE CONSTRAINT phone_unique IF NOT EXISTS FOR (p:Phone) REQUIRE p.phoneNumber IS UNIQUE",
    ]
    for c in constraints:
        tx.run(c)

with driver.session() as session:
    session.execute_write(create_constraints)
    print("Đã tạo constraint xong")

Đã tạo constraint xong


In [16]:
with driver.session() as session:
    result = session.run("SHOW CONSTRAINTS")
    for record in result:
        print(record)

<Record id=6 name='client_id' type='NODE_PROPERTY_UNIQUENESS' entityType='NODE' labelsOrTypes=['Client'] properties=['id'] enforcedLabel=None ownedIndex='client_id' propertyType=None>
<Record id=3 name='constraint_daa21259' type='NODE_PROPERTY_UNIQUENESS' entityType='NODE' labelsOrTypes=['_Bloom_Perspective_'] properties=['id'] enforcedLabel=None ownedIndex='constraint_daa21259' propertyType=None>
<Record id=14 name='email_unique' type='NODE_PROPERTY_UNIQUENESS' entityType='NODE' labelsOrTypes=['Email'] properties=['email'] enforcedLabel=None ownedIndex='email_unique' propertyType=None>
<Record id=8 name='merchant_id' type='NODE_PROPERTY_UNIQUENESS' entityType='NODE' labelsOrTypes=['Merchant'] properties=['id'] enforcedLabel=None ownedIndex='merchant_id' propertyType=None>
<Record id=16 name='phone_unique' type='NODE_PROPERTY_UNIQUENESS' entityType='NODE' labelsOrTypes=['Phone'] properties=['phoneNumber'] enforcedLabel=None ownedIndex='phone_unique' propertyType=None>
<Record id=12 nam

In [17]:
import pandas as pd

tx_df = pd.read_csv('paysim_sample.csv')
identity_df = pd.read_csv('identities.csv')

print(f"Số giao dịch: {len(tx_df)}")
print(f"Số identity: {len(identity_df)}")

BATCH_SIZE = 2000  # nạp theo lô, không nạp từng dòng một

Số giao dịch: 258213
Số identity: 392614


In [20]:
def load_transaction_batch(tx, rows):
    tx.run("""
    UNWIND $rows AS row
    MERGE (orig:Account {id: row.nameOrig})
      ON CREATE SET orig:Client, orig.accountType = 'Client'
    FOREACH (_ IN CASE WHEN row.nameOrig STARTS WITH 'M' THEN [1] ELSE [] END |
      SET orig:Merchant REMOVE orig:Client SET orig.accountType = 'Merchant'
    )
    MERGE (dest:Account {id: row.nameDest})
      ON CREATE SET dest:Client, dest.accountType = 'Client'
    FOREACH (_ IN CASE WHEN row.nameDest STARTS WITH 'M' THEN [1] ELSE [] END |
      SET dest:Merchant REMOVE dest:Client SET dest.accountType = 'Merchant'
    )
    CREATE (t:Transaction {
      id: randomUUID(),
      step: toInteger(row.step),
      amount: toFloat(row.amount),
      oldBalanceOrig: toFloat(row.oldbalanceOrg),
      newBalanceOrig: toFloat(row.newbalanceOrig),
      oldBalanceDest: toFloat(row.oldbalanceDest),
      newBalanceDest: toFloat(row.newbalanceDest),
      isFraud: toInteger(row.isFraud),
      isFlaggedFraud: toInteger(row.isFlaggedFraud)
    })
    WITH t, row, orig, dest
    CALL apoc.create.addLabels(t, [row.type]) YIELD node
    CREATE (orig)-[:PERFORMED]->(node)
    CREATE (node)-[:TO]->(dest)
    """, rows=rows)

with driver.session() as session:
    for i in range(0, len(tx_df), BATCH_SIZE):
        batch = tx_df.iloc[i:i+BATCH_SIZE].to_dict('records')
        session.execute_write(load_transaction_batch, batch)
        if i % (BATCH_SIZE * 10) == 0:
            print(f"Đã nạp {i}/{len(tx_df)} giao dịch")

print("Hoàn thành nạp Transaction")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. apoc.create.addLabels is deprecated. It is replaced by Cypher's dynamic labels; `SET n:$(labels)`..", position=<SummaryInputPosition line=25, column=5, offset=1014>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1014, 'line': 25, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    UNWIND $rows AS row\n    MERGE (orig:Account {id: row.nameOrig})\n      ON CREATE SET orig:Client, orig.accountType = 'Client'\n    FOREACH (_ IN CASE WHEN row.nameOrig STARTS WITH 'M' THEN [1] ELSE [] END |\n      SET orig:Merchant REMOVE orig:Client SET orig.accountType = 'Merchant'\n    )\n   

Đã nạp 0/258213 giao dịch


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. apoc.create.addLabels is deprecated. It is replaced by Cypher's dynamic labels; `SET n:$(labels)`..", position=<SummaryInputPosition line=25, column=5, offset=1014>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1014, 'line': 25, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    UNWIND $rows AS row\n    MERGE (orig:Account {id: row.nameOrig})\n      ON CREATE SET orig:Client, orig.accountType = 'Client'\n    FOREACH (_ IN CASE WHEN row.nameOrig STARTS WITH 'M' THEN [1] ELSE [] END |\n      SET orig:Merchant REMOVE orig:Client SET orig.accountType = 'Merchant'\n    )\n   

Đã nạp 20000/258213 giao dịch


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. apoc.create.addLabels is deprecated. It is replaced by Cypher's dynamic labels; `SET n:$(labels)`..", position=<SummaryInputPosition line=25, column=5, offset=1014>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1014, 'line': 25, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    UNWIND $rows AS row\n    MERGE (orig:Account {id: row.nameOrig})\n      ON CREATE SET orig:Client, orig.accountType = 'Client'\n    FOREACH (_ IN CASE WHEN row.nameOrig STARTS WITH 'M' THEN [1] ELSE [] END |\n      SET orig:Merchant REMOVE orig:Client SET orig.accountType = 'Merchant'\n    )\n   

KeyboardInterrupt: 

In [19]:
with driver.session() as session:
    result = session.run("SHOW CONSTRAINTS")
    for r in result:
        print(r)

<Record id=6 name='client_id' type='NODE_PROPERTY_UNIQUENESS' entityType='NODE' labelsOrTypes=['Client'] properties=['id'] enforcedLabel=None ownedIndex='client_id' propertyType=None>
<Record id=3 name='constraint_daa21259' type='NODE_PROPERTY_UNIQUENESS' entityType='NODE' labelsOrTypes=['_Bloom_Perspective_'] properties=['id'] enforcedLabel=None ownedIndex='constraint_daa21259' propertyType=None>
<Record id=14 name='email_unique' type='NODE_PROPERTY_UNIQUENESS' entityType='NODE' labelsOrTypes=['Email'] properties=['email'] enforcedLabel=None ownedIndex='email_unique' propertyType=None>
<Record id=8 name='merchant_id' type='NODE_PROPERTY_UNIQUENESS' entityType='NODE' labelsOrTypes=['Merchant'] properties=['id'] enforcedLabel=None ownedIndex='merchant_id' propertyType=None>
<Record id=16 name='phone_unique' type='NODE_PROPERTY_UNIQUENESS' entityType='NODE' labelsOrTypes=['Phone'] properties=['phoneNumber'] enforcedLabel=None ownedIndex='phone_unique' propertyType=None>
<Record id=12 nam